**This notebook contains code to extract relevant data from the metabolites xml file and match to the corresponding H-NMR spectra.**

**1. Build a list of HMDB ID's that have 1D HNMR data, based on filenames.**

In [12]:
import os, re, json, csv

root = "data/hmdb_nmr_peak_lists"

id_re = re.compile(r'HMDB[\s_-]*(\d{5,7})', re.IGNORECASE)

def norm_id(d): return f"HMDB{int(d):07d}"

def looks_like_oned_h1(filepath):
    """
    Heuristics:
      1) filenames containing 'nmroned' -> accept
      2) otherwise, peek header/body:
         - reject if it mentions 'F1' AND 'F2' (2D)
         - accept if it contains '1H' and not '13C'/'15N' in axis/nucleus lines
         - quick numeric sniff: lines with two floats -> likely 2D; single float ppm -> likely 1D
    """
    fn = os.path.basename(filepath).lower()
    if "nmroned" in fn:
        return True
    if "nmrtwod" in fn:
        return False

    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read(20000)
    except Exception:
        return False

    low = text.lower()
    if "f1" in low and "f2" in low:
        return False
    # nucleus hints
    if "13c" in low or "15n" in low:
        return False
    if "1h" in low or "proton" in low:
        pass  # keep checking

    # simple numeric sniff: count lines with 1 vs 2+ floats
    one_cols = two_plus_cols = 0
    for line in text.splitlines():
        toks = re.findall(r'[-+]?\d*\.\d+|\d+', line)
        if not toks: 
            continue
        # ignore very long lines (headers)
        if len(toks) == 1:
            one_cols += 1
        elif len(toks) >= 2:
            two_plus_cols += 1
        if one_cols + two_plus_cols > 50:
            break
    # If mostly single-column numbers -> likely 1D peak list
    return one_cols >= max(10, two_plus_cols * 2)

keep_ids = set()
rows = []
excluded = []
total_txt = 0

for dirpath, _, files in os.walk(root):
    for fn in files:
        if not fn.lower().endswith(".txt"):
            continue
        total_txt += 1
        full = os.path.join(dirpath, fn)

        if not looks_like_oned_h1(full):
            excluded.append(os.path.relpath(full, root))
            continue

        # find HMDB id (filename first, then content)
        m = id_re.search(fn)
        digits = None
        if not m:
            try:
                with open(full, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read(20000)
                m = id_re.search(text)
            except Exception:
                m = None

        if m:
            digits = m.group(1)
            keep_ids.add(norm_id(digits))
            rows.append([os.path.relpath(full, root), norm_id(digits), "oned_h1"])
        else:
            excluded.append(os.path.relpath(full, root))

with open("keep_ids_oned_h1.json", "w") as f:
    json.dump(sorted(keep_ids), f, indent=2)

with open("oned_h1_file_map.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["file","hmdb_id","type"]); w.writerows(rows)

'''with open("excluded_non_oned_or_noid.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(excluded))'''

print("Total .txt files scanned:", total_txt)
print("Unique HMDB IDs with 1D 1H data:", len(keep_ids))
print("Example kept IDs:", list(sorted(keep_ids))[:10])



Total .txt files scanned: 1896
Unique HMDB IDs with 1D 1H data: 892
Example kept IDs: ['HMDB0000001', 'HMDB0000002', 'HMDB0000005', 'HMDB0000008', 'HMDB0000010', 'HMDB0000011', 'HMDB0000012', 'HMDB0000014', 'HMDB0000016', 'HMDB0000017']


**2. Match existing 1D H-NMR spectra files to relevant information in the metabolite xml file.**

In [13]:
from lxml import etree as ET
import csv, json

xml_path = "data/hmdb_metabolites/hmdb_metabolites.xml"          # or .xml.gz if gzipped (use gzip.open)
ids_path = "keep_ids_oned_h1.json"
out_csv  = "hmdb_subset_classes.csv"

with open(ids_path) as f:
    keep_ids = set(json.load(f))

NS = "{http://www.hmdb.ca}"  # HMDB default namespace

def txt(parent, tag):
    """Safe text getter for direct child"""
    if parent is None: 
        return ""
    el = parent.find(NS + tag)
    return el.text.strip() if el is not None and el.text else ""

# Stream on end-of-element for <metabolite> to keep memory low
context = ET.iterparse(xml_path, events=("end",), tag=NS + "metabolite")

with open(out_csv, "w", newline="", encoding="utf-8") as fout:
    w = csv.writer(fout)
    w.writerow(["accession","name","kingdom","super_class","class","sub_class"])

    for _, metab in context:
        accession = txt(metab, "accession")
        if accession in keep_ids:
            name = txt(metab, "name")
            taxonomy = metab.find(NS + "taxonomy")
            row = [
                accession,
                name,
                txt(taxonomy, "kingdom"),
                txt(taxonomy, "super_class"),
                txt(taxonomy, "class"),
                txt(taxonomy, "sub_class"),   # <- the level you’ll use most
            ]
            w.writerow(row)

        # --- free memory ---
        metab.clear()
        # remove processed siblings from the tree to keep memory bounded
        while metab.getprevious() is not None:
            del metab.getparent()[0]

print(f"Done. Wrote classifications for {out_csv}")


Done. Wrote classifications for hmdb_subset_classes.csv


**Convert the CSV to NumPy array**

In [15]:
import pandas as pd

df = pd.read_csv("hmdb_subset_classes.csv")

# Inspect
print(df.head())

# Convert to NumPy array
arr = df.to_numpy()

print(arr.shape)
print(arr[:5])



     accession                   name            kingdom  \
0  HMDB0000001      1-Methylhistidine  Organic compounds   
1  HMDB0000002     1,3-Diaminopropane  Organic compounds   
2  HMDB0000005     2-Ketobutyric acid  Organic compounds   
3  HMDB0000008  2-Hydroxybutyric acid  Organic compounds   
4  HMDB0000010       2-Methoxyestrone  Organic compounds   

                       super_class                             class  \
0    Organic acids and derivatives  Carboxylic acids and derivatives   
1       Organic nitrogen compounds          Organonitrogen compounds   
2    Organic acids and derivatives        Keto acids and derivatives   
3    Organic acids and derivatives     Hydroxy acids and derivatives   
4  Lipids and lipid-like molecules  Steroids and steroid derivatives   

                                sub_class  
0    Amino acids, peptides, and analogues  
1                                  Amines  
2  Short-chain keto acids and derivatives  
3     Alpha hydroxy acids and 

**Check how many unique sub_classes there are.**

In [17]:
df = pd.read_csv("hmdb_subset_classes.csv")

print("Number of unique sub_classes:", df["sub_class"].nunique())
print(df["sub_class"].value_counts())


Number of unique sub_classes: 144
sub_class
Amino acids, peptides, and analogues         152
Fatty acids and conjugates                    83
Carbohydrates and carbohydrate conjugates     73
Benzoic acids and derivatives                 28
Purines and purine derivatives                26
                                            ... 
Thiazoles                                      1
Phenylacetamides                               1
Benzothiadiazines                              1
Tyrosols and derivatives                       1
Ergostane steroids                             1
Name: count, Length: 144, dtype: int64


**Export all sub_classes and number of counts to a txt file**

In [21]:
import pandas as pd

df = pd.read_csv("hmdb_subset_classes.csv")

# Count entries per sub_class, sort descending
counts = df["sub_class"].value_counts()

# Save to file
with open("unique_subclasses_counts.txt", "w", encoding="utf-8") as f:
    for sub_class, count in counts.items():
        f.write(f"{sub_class}\t{count}\n")

print(f"Wrote {len(counts)} unique sub_classes with counts to unique_subclasses_counts.txt")


Wrote 144 unique sub_classes with counts to unique_subclasses_counts.txt


**Create a new category called "group" to combine some sub_classes that are biologically and spectroscopically similar**

In [23]:
import pandas as pd

# 1) Load your subset
df = pd.read_csv("hmdb_subset_classes.csv")  # must contain a "sub_class" column

# 2) Broad groups (edit these labels if you prefer)
group_map = {
    # Amino acids / peptides
    "Amino acids, peptides, and analogues": "Amino acids & peptides",
    "Hybrid peptides": "Amino acids & peptides",
    "Guanidines": "Amino acids & peptides",

    # Carbohydrates
    "Carbohydrates and carbohydrate conjugates": "Carbohydrates",
    "Alcohols and polyols": "Carbohydrates / polyols",
    "Isosorbides": "Carbohydrates / polyols",

    # Lipids (fatty acids, glycerolipids, phospholipids, eicosanoids, etc.)
    "Fatty acids and conjugates": "Lipids",
    "Fatty acid esters": "Lipids",
    "Fatty alcohols": "Lipids",
    "Lineolic acids and derivatives": "Lipids",
    "Eicosanoids": "Lipids",
    "Triradylcglycerols": "Lipids",               # triacylglycerols (HMDB spelling)
    "Glycerophosphocholines": "Lipids (phospholipids)",
    "Glycerophosphates": "Lipids (phospholipids)",
    "Phosphosphingolipids": "Lipids (phospholipids)",
    "Quinone and hydroquinone lipids": "Lipids",
    "Fatty acyl glycosides": "Lipids",

    # Steroids (incl. bile acids & vitamin D)
    "Steroids and steroid derivatives": "Steroids",
    "Bile acids, alcohols and derivatives": "Steroids",
    "Cholestane steroids": "Steroids",
    "Androstane steroids": "Steroids",
    "Estrane steroids": "Steroids",
    "Pregnane steroids": "Steroids",
    "Stigmastanes and derivatives": "Steroids",
    "Hydroxysteroids": "Steroids",
    "Oxosteroids": "Steroids",
    "Sulfated steroids": "Steroids",
    "Steroid esters": "Steroids",
    "Steroid lactones": "Steroids",
    "Vitamin D and derivatives": "Steroids",
    "Ergostane steroids": "Steroids",

    # Nucleobases, nucleosides/tides & related heterocycles
    "Purines and purine derivatives": "Nucleobases & related",
    "Pyrimidines and pyrimidine derivatives": "Nucleobases & related",
    "Purine ribonucleotides": "Nucleobases & related",
    "Purine deoxyribonucleotides": "Nucleobases & related",
    "Purine 2'-deoxyribonucleosides": "Nucleobases & related",
    "Pyrimidine 2'-deoxyribonucleosides": "Nucleobases & related",
    "Pyrimidine ribonucleotides": "Nucleobases & related",
    "Pyrimidine nucleotide sugars": "Nucleobases & related",
    "Cyclic purine nucleotides": "Nucleobases & related",
    "Pterins and derivatives": "Nucleobases & related",
    "Alloxazines and isoalloxazines": "Nucleobases & related",

    # Aromatic carboxylic acids & phenolics (benzenoids, heteroaromatic acids, polyphenols)
    "Benzoic acids and derivatives": "Aromatic acids & phenolics",
    "Pyridinecarboxylic acids and derivatives": "Aromatic acids & phenolics",
    "Quinoline carboxylic acids": "Aromatic acids & phenolics",
    "Furoic acid and derivatives": "Aromatic acids & phenolics",
    "Phenylacetic acids": "Aromatic acids & phenolics",
    "Phenylpyruvic acid derivatives": "Aromatic acids & phenolics",
    "Cinnamic acids": "Aromatic acids & phenolics",
    "Hydroxycinnamic acids and derivatives": "Aromatic acids & phenolics",
    "Benzenediols": "Aromatic acids & phenolics",
    "1-hydroxy-4-unsubstituted benzenoids": "Aromatic acids & phenolics",
    "1-hydroxy-2-unsubstituted benzenoids": "Aromatic acids & phenolics",
    "Methoxyphenols": "Aromatic acids & phenolics",
    "Cresols": "Aromatic acids & phenolics",
    "Anisoles": "Aromatic acids & phenolics",
    "Methoxybenzenes": "Aromatic acids & phenolics",
    "Phenylpropanes": "Aromatic acids & phenolics",
    "Hydrolyzable tannins": "Aromatic acids & phenolics",
    "Flavones": "Flavonoids & polyphenols",
    "Flavans": "Flavonoids & polyphenols",
    "Flavonoid glycosides": "Flavonoids & polyphenols",
    "Isoflav-2-enes": "Flavonoids & polyphenols",
    "Hydroxycoumarins": "Flavonoids & polyphenols",
    "Tyrosols and derivatives": "Flavonoids & polyphenols",

    # Nitrogen (and S) heteroaromatics & related amines
    "Indoles": "N-heteroaromatics & amines",
    "Hydroxyindoles": "N-heteroaromatics & amines",
    "Indolyl carboxylic acids and derivatives": "N-heteroaromatics & amines",
    "Imidazoles": "N-heteroaromatics & amines",
    "Imidazolines": "N-heteroaromatics & amines",
    "Thiazoles": "N-heteroaromatics & amines",
    "Carbazoles": "N-heteroaromatics & amines",
    "Pyrrole carboxylic acids and derivatives": "N-heteroaromatics & amines",
    "Pyrrolidinylpyridines": "N-heteroaromatics & amines",
    "Hydropyridines": "N-heteroaromatics & amines",
    "Aniline and substituted anilines": "N-heteroaromatics & amines",
    "Phenethylamines": "N-heteroaromatics & amines",
    "Phenylbutylamines": "N-heteroaromatics & amines",
    "Quaternary ammonium salts": "Amines & ammonium salts",
    "Amines": "Amines & ammonium salts",
    "Aminoxides": "Amines & ammonium salts",

    # Aliphatic organic acids (non-aromatic)
    "Carboxylic acids": "Aliphatic organic acids",
    "Carboxylic acid derivatives": "Aliphatic organic acids",
    "Dicarboxylic acids and derivatives": "Aliphatic organic acids",
    "Tricarboxylic acids and derivatives": "Aliphatic organic acids",
    "Alpha hydroxy acids and derivatives": "Aliphatic organic acids",
    "Beta hydroxy acids and derivatives": "Aliphatic organic acids",
    "Short-chain hydroxy acids and derivatives": "Aliphatic organic acids",
    "Medium-chain hydroxy acids and derivatives": "Aliphatic organic acids",
    "Alpha-keto acids and derivatives": "Aliphatic organic acids",
    "Beta-keto acids and derivatives": "Aliphatic organic acids",
    "Gamma-keto acids and derivatives": "Aliphatic organic acids",
    "Short-chain keto acids and derivatives": "Aliphatic organic acids",
    "Medium-chain keto acids and derivatives": "Aliphatic organic acids",
    "Gamma butyrolactones": "Aliphatic organic acids",
    "Delta valerolactones": "Aliphatic organic acids",
    "3-phenoxypropionic acids": "Aliphatic organic acids",  # aromatic tail but aliphatic acid core
    "Carboximidic acids": "Aliphatic organic acids",

    # Terpenoids / isoprenoids (non-steroidal)
    "Monoterpenoids": "Terpenoids & isoprenoids",
    "Triterpenoids": "Terpenoids & isoprenoids",
    "Tetraterpenoids": "Terpenoids & isoprenoids",
    "Retinoids": "Terpenoids & isoprenoids",
    "Isoprenoid phosphates": "Terpenoids & isoprenoids",

    # Phosphate esters & organophosphorus
    "Phosphate esters": "Phosphates & organophosphorus",
    "Organic phosphonic acids": "Phosphates & organophosphorus",
    "Thiophosphoric acid esters": "Phosphates & organophosphorus",

    # Vitamins, cofactors & pigments
    "Corrinoids": "Vitamins & cofactors",
    "Pyridoxines": "Vitamins & cofactors",
    "Pyridoxamines": "Vitamins & cofactors",
    "Lipoic acids and derivatives": "Vitamins & cofactors",
    "Naphthoquinones": "Vitamins & cofactors",
    "Bilirubins": "Tetrapyrroles & pigments",

    # Simple aromatics / hydrocarbons / others
    "Toluenes": "Simple aromatics",
    "Halobenzenes": "Simple aromatics",
    "Biphenyls and derivatives": "Simple aromatics",
    "Diphenylmethanes": "Simple aromatics",
    "Benzyl alcohols": "Simple aromatics",
    "Benzylethers": "Simple aromatics",
    "Benzoyl derivatives": "Simple aromatics",
    "Pyridine carboxaldehydes": "Simple aromatics",
    "Dibenzazepines": "Simple aromatics",
    "Alkanes": "Hydrocarbons",

    # Sulfur-containing (non-aromatic)
    "Alkylthiols": "Organosulfur",
    "Sulfones": "Organosulfur",
    "Sulfinic acids": "Organosulfur",
    "Dialkylthioethers": "Organosulfur",
    "Organosulfonic acids and derivatives": "Organosulfur",
    "Arylsulfates": "Organosulfur (conjugates)",
    "Sulfinylbenzimidazoles": "Organosulfur (heterocycles)",

    # Odds & ends
    "Oximes": "Other",
    "Carbonyl compounds": "Other",
    "Ureas": "Other",
    "Piperidinones": "Other N-heterocycles",
    "Pyrrolidones": "Other N-heterocycles",
    "Quinone and hydroquinone lipids": "Lipids",  # already above but safe
}

# 3) Apply mapping; anything not listed -> "Other (unmapped)"
df["group"] = df["sub_class"].map(group_map).fillna("Other (unmapped)")

# 4) Save outputs
df.to_csv("hmdb_subset_with_groups.csv", index=False)
df["group"].value_counts().to_csv("group_counts.csv")

# 5) Help you refine: list any subclasses we didn't catch
unmapped = sorted(set(df["sub_class"].dropna()) - set(group_map))
pd.Series(unmapped, name="unmapped_sub_class").to_csv("UNMAPPED_subclasses.txt", index=False)



# 6) Print some summary stats
total = len(df)
mapped = (df["group"] != "Other (unmapped)").sum()
print(f"Total compounds: {total}")
print(f"Compounds assigned to a group: {mapped}")
print(f"Compounds still unmapped: {total - mapped}")

print("Wrote:")
print(" - hmdb_subset_with_groups.csv (added 'group' column)")
print(" - group_counts.csv (counts per group)")
print(" - UNMAPPED_subclasses.txt (review these to expand group_map)")


Total compounds: 890
Compounds assigned to a group: 813
Compounds still unmapped: 77
Wrote:
 - hmdb_subset_with_groups.csv (added 'group' column)
 - group_counts.csv (counts per group)
 - UNMAPPED_subclasses.txt (review these to expand group_map)
